# Verify STMUS

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import skimage.io as io

RAW_DATA_DIR = Path("/scratch/data/stmus_raw_20260826/Polito-Radboud-DeepLearningUS/")

In [ ]:
# Create a dataframe for the images. File path format:
# (BB|GM|TA)/(Healthy|Pathological)/(Images|Masks)/anon_{number:4d}_{frame_id:d}.png  # noqa: W505, E501

# NOTE: `number` is local to each muscle and echogenicity directory. It is NOT
# a patient ID. Patients come from the characteristics spreadsheets instead.
from ultrabench.process.stmus import (
    assign_patient_ids,
    collect_examples,
    load_characteristics,
)


def parse_image_path(path):
    parts = path.parts
    return {
        "number": int(parts[-1].split("_")[1]),
        "frame_id": int(parts[-1].split("_")[2].split(".")[0]),
        "echogenicity": parts[-3],
        "muscle": parts[-4],
        "image_path": path,
        "mask_path": path.parent.parent / "Masks" / parts[-1],
        "filename": f"{parts[-4].lower()}_{parts[-3].lower()}_{path.stem}",
    }


image_paths = sorted(RAW_DATA_DIR.glob("**/Images/*.png"))
df = pd.DataFrame([parse_image_path(p) for p in image_paths])

# Attach the resolved patient IDs and demographics, keyed on the filename.
examples = collect_examples(str(RAW_DATA_DIR))
df = df.merge(
    examples[["filename", "patient", "age", "height", "weight", "sex"]],
    on="filename",
    validate="one_to_one",
)
df.head()

In [ ]:
# Count the number of images and number of patients
print("Images:", df["image_path"].nunique())
print("Patients:", df["patient"].nunique())
print("Folder-local subject numbers (NOT patients):", df["number"].nunique())

assert len(df) == 8169
assert df["filename"].nunique() == 8169
assert df["patient"].nunique() == 1223

In [ ]:
# Check that all images and masks exist
assert df["image_path"].apply(lambda p: p.exists()).all()
assert df["mask_path"].apply(lambda p: p.exists()).all()

In [ ]:
# Check the number of channels, dimensions, and min/max values of the images
channels = []
heights = []
widths = []
min_value = -np.inf
max_value = np.inf
for path in df["image_path"]:
    image = io.imread(path)
    channels.append(image.shape[2] if image.ndim == 3 else 1)
    heights.append(image.shape[0])
    widths.append(image.shape[1])
    min_value = max(min_value, image.min())
    max_value = min(max_value, image.max())

print("Channels:", set(channels))
print("Heights: min = {:.2f}, mean = {:.2f}, max = {:.2f}".format(min(heights), np.mean(heights), max(heights)))
print("Widths: min = {:.2f}, mean = {:.2f}, max = {:.2f}".format(min(widths), np.mean(widths), max(widths)))
print("Min value:", min_value)
print("Max value:", max_value)

In [ ]:
# Check the number of channels and pixel values of the masks
mask_channels = set()
mask_values = set()
for path in df["mask_path"]:
    mask = io.imread(path)
    mask_channels.add(mask.shape[2 if mask.ndim == 3 else 1])
    mask_values.update(np.unique(mask))

print("Mask channels:", mask_channels)
print("Mask values:", mask_values)

In [ ]:
# Count the number of images per muscle and echogenicity
df.groupby(["muscle", "echogenicity"]).size()

In [ ]:
# Count the number of patients per muscle and echogenicity. The folder-local
# numbers are shown alongside to make the difference between them visible.
pd.DataFrame(
    {
        "patients": df.groupby(["muscle", "echogenicity"])["patient"].nunique(),
        "subject_numbers": df.groupby(["muscle", "echogenicity"])["number"].nunique(),
    }
)

## Verify patient identification

The subject number in the filename is local to each muscle and echogenicity
directory, so it cannot be used to group images by patient. The cells below
show the evidence for that, then check the patient IDs derived from the
`characteristics.xlsx` spreadsheets.

In [ ]:
# Evidence that the subject number is folder-local: where the same number
# appears in two muscle directories, the demographics almost never agree, and
# the recorded sex differs about half the time.
import itertools

characteristics = load_characteristics(str(RAW_DATA_DIR))
DEMOGRAPHICS = ["age", "height", "weight", "sex"]

rows = []
for a, b in itertools.combinations(["BB", "GM", "TA"], 2):
    left = characteristics[characteristics["muscle_code"] == a].set_index("number")
    right = characteristics[characteristics["muscle_code"] == b].set_index("number")
    shared = left.index.intersection(right.index)
    same = (left.loc[shared, DEMOGRAPHICS] == right.loc[shared, DEMOGRAPHICS]).all(
        axis=1
    )
    rows.append(
        {
            "pair": f"{a} vs {b}",
            "shared_numbers": len(shared),
            "same_demographics": int(same.sum()),
            "median_age_gap": (left.loc[shared, "age"] - right.loc[shared, "age"])
            .abs()[~same]
            .median(),
            "sex_differs": int(
                (left.loc[shared, "sex"] != right.loc[shared, "sex"])[~same].sum()
            ),
        }
    )

pd.DataFrame(rows)

In [ ]:
# Every image joins to a subject entry, and demographics are constant within a
# patient. Some spreadsheet entries have no images, which is expected.
patients = assign_patient_ids(characteristics)

assert df["patient"].notna().all()
assert df.groupby("patient")[DEMOGRAPHICS].nunique().max().max() == 1

entries_with_images = df.groupby(["muscle", "echogenicity", "number"]).ngroups
print("Subject entries in the spreadsheets:", len(characteristics))
print("Subject entries with images:", entries_with_images)
print("Entries without images:", len(characteristics) - entries_with_images)
print()
print("Muscles per patient:")
print(df.groupby("patient")["muscle"].nunique().value_counts().sort_index())
print()
images_per_patient = df.groupby("patient").size()
print(
    "Images per patient: min {}, median {:.0f}, max {}".format(
        images_per_patient.min(), images_per_patient.median(), images_per_patient.max()
    )
)

In [ ]:
# Independent cross-check. Subjects_mean_gray_level_zscore.xlsx records the
# same demographics for the subset the original paper analyzed. Its 1,283 rows
# are exactly the "1283 subjects" the paper reports, counted as one entry per
# muscle scanned rather than per person.
zscore_sheets = {"BB": 0, "GM": 1, "TA": 2}
blocks = []
for muscle_code, sheet in zscore_sheets.items():
    raw = pd.read_excel(
        RAW_DATA_DIR / "Subjects_mean_gray_level_zscore.xlsx",
        sheet_name=sheet,
        header=None,
    )
    for echogenicity, first_column in [("healthy", 0), ("pathological", 11)]:
        block = raw.iloc[
            2:,
            [
                first_column,
                first_column + 5,
                first_column + 6,
                first_column + 7,
                first_column + 8,
            ],
        ].copy()
        block.columns = ["number", *DEMOGRAPHICS]
        block = block[pd.to_numeric(block["number"], errors="coerce").notna()]
        blocks.append(block.assign(muscle_code=muscle_code, echogenicity=echogenicity))

zscore = pd.concat(blocks, ignore_index=True).astype({"number": int})
merged = zscore.merge(
    characteristics, on=["muscle_code", "echogenicity", "number"], suffixes=("_z", "_c")
)

agree = ((merged["age_z"].astype(float) - merged["age_c"]).abs() < 1e-4) & (
    merged["sex_z"] == merged["sex_c"]
)
print(f"Rows in the z-score spreadsheet: {len(zscore)}")
print(f"Demographics agree: {agree.sum()}/{len(merged)}")
assert len(zscore) == 1283
assert agree.all()

In [ ]:
# Reproduce the published split and confirm there is no patient overlap.
from sklearn.model_selection import GroupShuffleSplit

from ultrabench.process.utils import cap_train_split_by_patient

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(examples, groups=examples["patient"]))
train_df = examples.iloc[train_idx].reset_index(drop=True)
test_df = examples.iloc[test_idx].reset_index(drop=True)
train_df, test_df = cap_train_split_by_patient(train_df, test_df)

overlap = set(train_df["patient"]) & set(test_df["patient"])
print(f"train: {len(train_df)} images from {train_df['patient'].nunique()} patients")
print(f"test:  {len(test_df)} images from {test_df['patient'].nunique()} patients")
print("Patients in both splits:", len(overlap))

assert not overlap
assert len(train_df) == 1000
assert len(test_df) == 7169

In [ ]:
# Visualize some examples of each muscle and echogenicity (row: echogenicity,
# column: muscle)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for i in range(df["echogenicity"].nunique()):
    for j in range(df["muscle"].nunique()):
        example_image_path = df[
            (df["echogenicity"] == df["echogenicity"].unique()[i])
            & (df["muscle"] == df["muscle"].unique()[j])
        ]["image_path"].iloc[0]
        example_image = plt.imread(example_image_path)
        axes[i, j].imshow(example_image, cmap="gray")
        axes[i, j].set_title(
            f"{df['echogenicity'].unique()[i]} {df['muscle'].unique()[j]}"
        )
        axes[i, j].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Visualize some examples of each muscle and echogenicity (row: echogenicity,
# column: muscle) with masks overlaid
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for i in range(df["echogenicity"].nunique()):
    for j in range(df["muscle"].nunique()):
        example_image_path = df[
            (df["echogenicity"] == df["echogenicity"].unique()[i])
            & (df["muscle"] == df["muscle"].unique()[j])
        ]["image_path"].iloc[0]
        example_mask_path = df[
            (df["echogenicity"] == df["echogenicity"].unique()[i])
            & (df["muscle"] == df["muscle"].unique()[j])
        ]["mask_path"].iloc[0]
        example_image = plt.imread(example_image_path)
        example_mask = plt.imread(example_mask_path)
        axes[i, j].imshow(example_image, cmap="gray")
        axes[i, j].imshow(example_mask, alpha=0.5)
        axes[i, j].set_title(
            f"{df['echogenicity'].unique()[i]} {df['muscle'].unique()[j]}"
        )
        axes[i, j].axis("off")

plt.tight_layout()
plt.show()